In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Reprezentace kvantových počítačů pro Transpiler


<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících závislostí.
Doporučujeme používat tyto verze nebo novější.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Aby mohl Transpiler převést abstraktní Circuit na ISA circuit, který lze spustit na konkrétní QPU (quantum processing unit), potřebuje určité informace o QPU. Tyto informace se nacházejí na dvou místech: v objektu `BackendV2` (nebo starším `BackendV1`), na který plánuješ odesílat úlohy, a v atributu `Target` daného backendu.

- [`Target`](../api/qiskit/qiskit.transpiler.Target) obsahuje všechna relevantní omezení zařízení, jako jsou jeho nativní basis gates, konektivita Qubitů a informace o pulzech nebo časování.
- [`Backend`](../api/qiskit/qiskit.providers.BackendV2) má `Target` jako výchozí součást, obsahuje další informace -- například [`InstructionScheduleMap`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.pulse.InstructionScheduleMap), a poskytuje rozhraní pro odesílání úloh kvantových Circuit.

Můžeš také explicitně poskytnout informace, které má Transpiler použít -- například pokud máš specifický případ použití nebo se domníváš, že tyto informace pomohou Transpileru vygenerovat optimalizovanější Circuit.

Přesnost, s níž Transpiler vytváří nejvhodnější Circuit pro konkrétní hardware, závisí na tom, kolik informací o svých omezeních má daný `Target` nebo `Backend` k dispozici.

> **Note:** Protože mnoho základních algoritmů transpilace je stochastických, není zaručeno, že bude nalezen lepší circuit.

Tato stránka ukazuje několik příkladů předávání informací o QPU do Transpileru. Tyto příklady používají target z mockového backendu [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

<span id="default-config"></span>
## Výchozí konfigurace
Nejjednodušší způsob použití Transpileru je poskytnout veškeré informace o QPU předáním `Backend` nebo `Target`. Abys lépe pochopil/a, jak Transpiler funguje, sestav Circuit a proveď transpilaci s různými informacemi, jak je popsáno níže.

Importuj potřebné knihovny a vytvoř instanci QPU:
Aby mohl Transpiler převést abstraktní Circuit na ISA circuit, který lze spustit na konkrétním procesoru, potřebuje určité informace o procesoru. Tyto informace jsou obvykle uloženy v [`Backend`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.Backend#backend) nebo [`Target`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Target#target) předaném Transpileru a žádné další informace nejsou potřeba. Můžeš ale také explicitně poskytnout informace, které má Transpiler použít -- například pokud máš specifický případ použití nebo se domníváš, že tyto informace pomohou Transpileru vygenerovat optimalizovanější Circuit.

Toto téma ukazuje několik příkladů předávání informací Transpileru. Tyto příklady používají target z mockového backendu [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
target = backend.target

Ukázkový Circuit používá instanci [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) z knihovny Circuit Qikit.

In [2]:
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)

qc.draw("mpl")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg)

Tento příklad používá výchozí nastavení pro transpilaci do `target` backendu, který poskytuje veškeré informace potřebné k převodu Circuit do podoby, jež bude na backendu spustitelná.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=target, seed_transpiler=12345
)
qc_t_target = pass_manager.run(qc)
qc_t_target.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg)

Tento příklad je využíván v dalších částech tohoto tématu k ilustraci toho, že coupling map a basis gates jsou zásadní informace, které je třeba předat Transpileru pro optimální sestavení Circuit. QPU obvykle dokáže pro ostatní nepředané informace, jako je časování a plánování, zvolit výchozí nastavení.

## Coupling map


Coupling map je graf znázorňující, které Qubity jsou propojeny, a tedy mají mezi sebou dvouqubitové Gate. Někdy je tento graf orientovaný, což znamená, že dvouqubitové Gate mohou být aplikovány pouze v jednom směru. Transpiler však dokáže směr Gate vždy otočit přidáním dalších jednoqubitových Gate. Abstraktní kvantový Circuit lze vždy reprezentovat na tomto grafu, i když je jeho konektivita omezená, a to vložením SWAP Gate pro přesun kvantové informace.

Qubity z našich abstraktních Circuit se nazývají _virtuální Qubity_ a ty v coupling map jsou _fyzické Qubity_. Transpiler zajišťuje mapování mezi virtuálními a fyzickými Qubity. Jedním z prvních kroků transpilace je fáze _layout_, která toto mapování provádí.

> **Note:** Ačkoli je fáze routování propojena s fází _layout_ -- která vybírá skutečné Qubity -- toto téma je pro jednoduchost ve výchozím nastavení považuje za oddělené fáze. Kombinace routování a layoutu se nazývá _qubit mapping_. Více o těchto fázích se dozvíš v tématu [Fáze Transpileru](transpiler-stages).

Předej argument `coupling_map` pro zobrazení jeho vlivu na Transpiler:

In [4]:
coupling_map = target.build_coupling_map()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv0 = pass_manager.run(qc)
qc_t_cm_lv0.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg)

Jak je vidět výše, bylo vloženo několik SWAP Gate (každá se skládá ze tří CX Gate), což způsobí na současných zařízeních velké množství chyb. Chceš-li zjistit, které Qubity jsou vybrány na skutečné topologii Qubitů, použij `plot_circuit_layout` z Qiskit Visualizations:

In [5]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv0, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg)

Toto ukazuje, že naše virtuální Qubity 0–11 byly triviálně namapovány na řadu fyzických Qubitů 0–11. Vraťme se k výchozímu nastavení (`optimization_level=1`), které používá `VF2Layout`, je-li potřeba jakékoli routování.

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv1 = pass_manager.run(qc)
qc_t_cm_lv1.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg)

Nyní nejsou vloženy žádné SWAP Gate a vybrané fyzické Qubity jsou stejné jako při použití třídy `target`.

In [7]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv1, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg)

Nyní je layout v kruhu. Protože tento layout respektuje konektivitu Circuit, nejsou vloženy žádné SWAP Gate, což poskytuje mnohem lepší Circuit pro spuštění.

## Basis gates


Každý kvantový počítač podporuje omezený instrukční set, nazývaný jeho _basis gates_. Každá Gate v Circuit musí být přeložena na prvky tohoto setu. Tento set by měl obsahovat jednoqubitové a dvouqubitové Gate, které tvoří univerzální set Gate -- to znamená, že jakákoli kvantová operace může být rozložena do těchto Gate. Toto zajišťuje [BasisTranslator](../api/qiskit/qiskit.transpiler.passes.BasisTranslator) a basis gates lze jako argument Transpileru zadat explicitně.

In [8]:
basis_gates = list(target.operation_names)
print(basis_gates)

['sx', 'switch_case', 'x', 'if_else', 'measure', 'for_loop', 'delay', 'ecr', 'id', 'reset', 'rz']


The default single-qubit gates on _ibm_sherbrooke_ are `rz`, `x`, and `sx`, and the default two-qubit gate is `ecr` (echoed cross-resonance). CX gates are constructed from `ecr` gates, so on some QPUs `ecr` is specified as the two-qubit basis gate, while on others `cx` is the default. The `ecr` gate is the _entangling_ part of the `cx` gate. In addition to the control gates, there are also `delay` and `measurement` instructions.


<Admonition type="note">
    QPUs have default basis gates, but you can choose whatever gates you want, as long as you provide the instruction or add pulse gates (see [Create transpiler passes.](custom-transpiler-pass)) The default basis gates are those that calibrations have been done for on the QPU, so no further instruction/pulse gates need to be provided. For example, on some QPUs `cx` is the default two-qubit gate and `ecr` on others. See the list of possible [native gates and operations](/docs/guides/qpu-information#native-gates) for more details.
</Admonition>

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    seed_transpiler=12345,
)
qc_t_cm_bg = pass_manager.run(qc)
qc_t_cm_bg.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg" alt="Output of the previous code cell" />

Výchozí jednoqubitové Gate na _ibm_sherbrooke_ jsou `rz`, `x` a `sx`, výchozí dvouqubitová Gate je `ecr` (echoed cross-resonance). CX Gate jsou sestaveny z `ecr` Gate, takže na některých QPU je `ecr` zadána jako dvouqubitová basis gate, zatímco na jiných je výchozí `cx`. Gate `ecr` je _entanglující_ částí Gate `cx`. Kromě řídících Gate existují také instrukce `delay` a `measurement`.

> **Note:** QPU mají výchozí basis gates, ale můžeš si zvolit libovolné Gate -- za předpokladu, že poskytneš příslušnou instrukci nebo přidáš pulzní Gate (viz [Tvorba vlastních průchodů Transpileru.](custom-transpiler-pass)) Výchozí basis gates jsou ty, pro které byly na QPU provedeny kalibrace, takže není třeba poskytovat žádné další instrukce ani pulzní Gate. Například na některých QPU je výchozí dvouqubitová Gate `cx`, na jiných `ecr`. Viz seznam možných [nativních Gate a operací](/guides/qpu-information#native-gates) pro více podrobností.

In [10]:
target["ecr"][(1, 0)]

InstructionProperties(duration=5.333333333333332e-07, error=0.007494257741828603)

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg)

Všimni si, že objekty `CXGate` byly rozloženy na Gate `ecr` a jednoqubitové basis gates.
## Chybovosti zařízení
Třída `Target` může obsahovat informace o chybovostech operací na zařízení.
Například následující kód načte vlastnosti Gate echoed cross-resonance (ECR) mezi Qubitem 1 a 0 (všimni si, že Gate ECR je orientovaná):

In [11]:
from qiskit.transpiler import Target
from qiskit.circuit.controlflow import IfElseOp, SwitchCaseOp, ForLoopOp

err_targ = Target.from_configuration(
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    num_qubits=target.num_qubits,
    custom_name_mapping={
        "if_else": IfElseOp,
        "switch_case": SwitchCaseOp,
        "for_loop": ForLoopOp,
    },
)

for i, (op, qargs) in enumerate(target.instructions):
    if op.name in basis_gates:
        err_targ[op.name][qargs] = target.instruction_properties(i)

Transpile with our new target `err_targ` as the target:

In [12]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=err_targ, seed_transpiler=12345
)
qc_t_cm_bg_et = pass_manager.run(qc)
qc_t_cm_bg_et.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/f1e270c4-e2cc-487e-a050-4180bc321b0b-0.svg" alt="Output of the previous code cell" />

Výstup zobrazuje dobu trvání Gate (v sekundách) a její chybovost. Chceš-li zpřístupnit informace o chybách Transpileru, sestav model target s `basis_gates` a `coupling_map` z výše uvedeného kódu a naplň jej hodnotami chyb z backendu `FakeSherbrooke`.